In [ ]:
!pip install pandas numpy sentence-transformers

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import re

<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
df = pd.read_csv('/content/data.csv')

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
texts = df.iloc[:,0].dropna().astype(str).tolist()

print("Raw Sample:", texts[:3])

Raw Sample: ['Senior Cognos Developer - IBM Cognos Framework - Baltimore, MD', 'Sales Reps (EMEA)', 'Junior Product Support Engineer - Funnelback ']


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9 ]', '', text)  # remove special chars
    text = re.sub(r'\s+', ' ', text).strip()
    return text

texts_clean = [clean_text(t) for t in texts]

print("Clean Sample:", texts_clean[:3])


Clean Sample: ['senior cognos developer ibm cognos framework baltimore md', 'sales reps emea', 'junior product support engineer funnelback']


In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(texts_clean)

print("Embedding Shape:", embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Shape: (12725, 384)


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
embeddings_scaled = scaler.fit_transform(embeddings)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
SEQ_LEN = 10   # sequence length (tunable)

def create_3d_matrix(data, seq_len):
    sequences = []

    for i in range(len(data) - seq_len):
        seq = data[i:i+seq_len]
        sequences.append(seq)

    return np.array(sequences)

matrix_3d = create_3d_matrix(embeddings_scaled, SEQ_LEN)

print("Final 3D Matrix Shape:", matrix_3d.shape)

Final 3D Matrix Shape: (12715, 10, 384)


In [ ]:
np.save("text_3d_matrix.npy", matrix_3d)

print("✅ 3D Matrix Saved Successfully!")

✅ 3D Matrix Saved Successfully!


In [ ]:
# ==============================
# IMPORTS
# ==============================
import numpy as np
import pandas as pd

# Load your 3D matrix
matrix_3d = np.load("text_3d_matrix.npy")

results = []

# ==============================
# METRIC COMPUTATION
# ==============================
for i in range(len(matrix_3d)):

    mat = matrix_3d[i]

    # Matrix multiplication
    result = np.dot(mat, mat.T)

    # ---------- METRICS ----------

    # Accuracy (stability-based)
    acc = np.mean(result) / (np.std(result) + 1e-5)
    acc = np.tanh(acc) * 100   # normalize to %

    # Cleanliness
    clean = (np.sum(mat != 0) / mat.size) * 100

    # Errors
    errors = (np.sum(np.isnan(mat)) / mat.size) * 100

    # Anomaly (outliers)
    anomaly = (np.sum(np.abs(mat) > 3) / mat.size) * 100

    # Reward (custom weighted %)
    reward = acc + clean - errors - anomaly
    reward = max(0, min(100, reward))  # clamp 0–100

    results.append([acc, clean, errors, anomaly, reward])

# ==============================
# CREATE TABLE
# ==============================
columns = ["Accuracy (%)", "Cleanliness (%)", "Error (%)", "Anomaly (%)", "Reward (%)"]

df_results = pd.DataFrame(results, columns=columns)

# Round values
df_results = df_results.round(2)

# ==============================
# SHOW OUTPUT
# ==============================
print(df_results.head())

# Save
df_results.to_csv("final_results_percentage.csv", index=False)

print("\n✅ Results saved as final_results_percentage.csv")

   Accuracy (%)  Cleanliness (%)  Error (%)  Anomaly (%)  Reward (%)
0     33.680000            100.0        0.0         0.44         100
1     37.930000            100.0        0.0         0.42         100
2     34.680000            100.0        0.0         0.39         100
3     32.169998            100.0        0.0         0.34         100
4     24.900000            100.0        0.0         0.36         100

✅ Results saved as final_results_percentage.csv


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
!pip install transformers sentence-transformers accelerate


In [10]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import StandardScaler

device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
import pandas as pd
df = pd.read_csv('/content/sample_data/california_housing_train.csv')
texts = df.iloc[:,0].dropna().astype(str).tolist()

In [7]:
import re

# Ensure 'texts' is defined from 'df' (if not already from previous cell)
# This line is added to handle NameError if previous cell's 'texts' is lost.
texts = df.iloc[:,0].dropna().astype(str).tolist()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9 ]', '', text)
    return text.strip()

texts = [clean_text(t) for t in texts]

In [11]:
# Define get_embeddings function
# This cell was executed to define the function, which was missing in the previous run.
def get_embeddings(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)

    embeddings = []

    for text in texts[:500]:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        # ✅ FIXED LINE
        emb = outputs.last_hidden_state.mean(dim=1).squeeze().to(torch.float32).cpu().numpy()

        embeddings.append(emb)

    return np.array(embeddings)

# Now, re-run the cell that calls get_embeddings
print("Loading Qwen...")
qwen_emb = get_embeddings("Qwen/Qwen2-0.5B")

print("Loading LLaMA...")
llama_emb = get_embeddings("sentence-transformers/all-mpnet-base-v2")

Loading Qwen...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def get_embeddings(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)

    embeddings = []

    for text in texts[:500]:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        # ✅ FIXED LINE
        emb = outputs.last_hidden_state.mean(dim=1).squeeze().to(torch.float32).cpu().numpy()

        embeddings.append(emb)

    return np.array(embeddings)

In [ ]:
# This cell was part of the previous 'get_embeddings' function and is now empty. The code has been moved to the preceding cell.

In [12]:
print("Loading Qwen...")
qwen_emb = get_embeddings("Qwen/Qwen2-0.5B")

print("Loading LLaMA...")
llama_emb = get_embeddings("sentence-transformers/all-mpnet-base-v2")

Loading Qwen...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading LLaMA...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
scaler = StandardScaler()

qwen_emb = scaler.fit_transform(qwen_emb)
llama_emb = scaler.fit_transform(llama_emb)

In [14]:
SEQ_LEN = 10

def create_3d(data):
    return np.array([data[i:i+SEQ_LEN] for i in range(len(data)-SEQ_LEN)])

qwen_3d = create_3d(qwen_emb)
llama_3d = create_3d(llama_emb)

In [15]:
import time

def compute_metrics(matrix_3d):

    results = []
    times = []

    for mat in matrix_3d:

        # ⏱️ START TIMER
        start_time = time.time()

        # Matrix multiplication
        result = np.dot(mat, mat.T)

        # ⏱️ END TIMER
        end_time = time.time()

        exec_time = (end_time - start_time) * 1000  # ms
        times.append(exec_time)

        # ---------- METRICS ----------
        acc = np.mean(result) / (np.std(result) + 1e-5)
        acc = np.tanh(acc) * 100

        clean = (np.sum(mat != 0) / mat.size) * 100
        errors = (np.sum(np.isnan(mat)) / mat.size) * 100
        anomaly = (np.sum(np.abs(mat) > 3) / mat.size) * 100

        reward = acc + clean - errors - anomaly
        reward = max(0, min(100, reward))

        results.append([acc, clean, errors, anomaly, reward, exec_time])

    return np.array(results), np.array(times)

In [16]:
qwen_results, qwen_time = compute_metrics(qwen_3d)
llama_results, llama_time = compute_metrics(llama_3d)

In [17]:
qwen_avg = qwen_results.mean(axis=0)
llama_avg = llama_results.mean(axis=0)

In [18]:
def normalize_time(time_array):
    max_t = np.max(time_array)
    min_t = np.min(time_array)

    # Invert: lower time → higher score
    norm = 100 * (1 - (time_array - min_t) / (max_t - min_t + 1e-5))
    return norm

In [19]:
qwen_time_score = normalize_time(qwen_time)
llama_time_score = normalize_time(llama_time)

In [20]:
columns = ["Accuracy (%)", "Cleanliness (%)", "Error (%)", "Anomaly (%)", "Reward (%)", "Time Score (%)"]

qwen_avg = np.hstack([qwen_results.mean(axis=0)[:5], qwen_time_score.mean()])
llama_avg = np.hstack([llama_results.mean(axis=0)[:5], llama_time_score.mean()])

comparison_df = pd.DataFrame(
    [qwen_avg, llama_avg],
    columns=columns,
    index=["Qwen", "LLaMA (proxy)"]
)

comparison_df = comparison_df.round(2)

print("\n🔥 FINAL TABLE WITH OPTIMIZATION TIME:\n")
print(comparison_df)


🔥 FINAL TABLE WITH OPTIMIZATION TIME:

               Accuracy (%)  Cleanliness (%)  Error (%)  Anomaly (%)  \
Qwen                  90.52            100.0        0.0         0.74   
LLaMA (proxy)         93.43            100.0        0.0         0.59   

               Reward (%)  Time Score (%)  
Qwen                100.0           99.76  
LLaMA (proxy)       100.0           90.44  
